# Un model de raționament local

Rulăm local Qwen3-1.7B, un model care poate scrie un raționament înainte de răspuns. Separăm gândurile de răspunsul final, comparăm modul cu și fără gândire și controlăm bugetul de gândire din cod.

Codul din acest notebook vine de pe pagina [Un model de raționament local](https://inteligentartificiala.ro/laboratoare/model-local-transformers/model-de-rationament/) a laboratorului [Un model de limbaj local cu Transformers](https://inteligentartificiala.ro/laboratoare/model-local-transformers/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro). Explicațiile fiecărui pas sunt pe pagină.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

In [ ]:
%pip install -q -U transformers accelerate

## Gândurile și răspunsul

În modul cu gândire, modelul scrie mai întâi un raționament între tokenurile `<think>` și `</think>`, apoi răspunsul. Modul se alege din șablonul de chat, cu argumentul `enable_thinking`. Codul de mai jos pune aceeași întrebare în ambele moduri și separă raționamentul de răspuns după poziția tokenului `</think>`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
end_think = tokenizer.convert_tokens_to_ids("</think>")

messages = [{"role": "user", "content": "How many numbers from 1 to 100 are divisible by 3 or by 5?"}]

for thinking in [True, False]:
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=thinking, return_dict=True, return_tensors="pt"
    ).to(model.device)
    # generation_config.json conține parametrii de eșantionare recomandați pentru modul cu gândire.
    output = model.generate(**inputs, max_new_tokens=8192)
    new_tokens = output[0, inputs["input_ids"].shape[1]:].tolist()

    # Raționamentul se termină la tokenul </think>. Fără gândire, el lipsește din răspuns.
    split = new_tokens.index(end_think) + 1 if end_think in new_tokens else 0
    reasoning = tokenizer.decode(new_tokens[:split], skip_special_tokens=True)
    answer = tokenizer.decode(new_tokens[split:], skip_special_tokens=True).strip()

    print(f"\nenable_thinking={thinking}: {split} tokenuri de gândire, {len(new_tokens) - split} în răspuns")
    print("Începutul raționamentului:", reasoning[:300])
    print("Finalul răspunsului:", answer[-120:])

## Bugetul de gândire

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")


def continue_text(text, max_new_tokens, **kwargs):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)


def answer_with_budget(question, max_thinking, min_thinking=0):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}], add_generation_prompt=True, enable_thinking=True, tokenize=False
    )
    used = 0
    while True:
        chunk, n = continue_text(text, max_thinking - used, stop_strings=["</think>"], tokenizer=tokenizer)
        used += n
        text += chunk.removesuffix("</think>")
        finished = chunk.endswith("</think>")
        if finished and used < min_thinking:
            text += "\nWait"  # modelul voia să se oprească prea devreme, așa că îl lăsăm să mai gândească
            continue
        if not finished:
            text += "\n\nI must give the final answer now.\n"  # bugetul s-a terminat
        break
    answer, _ = continue_text(text + "</think>\n\n", 1024)
    return used, answer.strip()


question = "How many numbers from 1 to 100 are divisible by 3 or by 5? Reply with the number only."
for max_thinking in [32, 256, 2048]:
    used, answer = answer_with_budget(question, max_thinking)
    print(f"buget maxim {max_thinking:4d}: {used:4d} tokenuri de gândire, răspuns: {answer[-40:]!r}")

used, answer = answer_with_budget(question, max_thinking=4096, min_thinking=3000)
print(f"buget minim 3000: {used} tokenuri de gândire, răspuns: {answer[-40:]!r}")